# IMDB Sentiment Analysis — Final Kaggle Submission

Train the selected word-level TF-IDF and logistic-regression model on all cleaned labeled reviews, then generate `submission.csv` for the Kaggle competition. The notebook runs both locally and in a Kaggle Notebook with the competition data attached.

In [ ]:
from pathlib import Path

from bs4 import BeautifulSoup
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

## Locate the competition files

On Kaggle, attach the **Bag of Words Meets Bags of Popcorn** competition data before running. Locally, the notebook uses `data/raw`.

In [ ]:
def find_data_file(filename: str) -> Path:
    local_roots = [Path.cwd(), Path.cwd().parent]
    local_candidates = [root / "data" / "raw" / filename for root in local_roots]
    for path in local_candidates:
        if path.is_file():
            return path

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        matches = sorted(kaggle_input.rglob(filename))
        if matches:
            return matches[0]

    raise FileNotFoundError(
        f"Could not find {filename}. Attach the competition data on Kaggle "
        "or place it in data/raw locally."
    )


train_path = find_data_file("labeledTrainData.tsv.zip")
test_path = find_data_file("testData.tsv.zip")
print(f"Training data: {train_path}")
print(f"Test data: {test_path}")

## Load and clean the reviews

Exact duplicate reviews are removed from the labeled training set. Test rows are never removed because every test ID must appear in the submission.

In [ ]:
def clean_review_text(reviews: pd.Series) -> pd.Series:
    cleaned = reviews.map(
        lambda review: BeautifulSoup(str(review), "html.parser").get_text(" ", strip=True)
    )
    return cleaned.str.replace(r"\s+", " ", regex=True)


train = pd.read_csv(train_path, sep="\t", compression="zip")
test = pd.read_csv(test_path, sep="\t", compression="zip")

train_clean = train.copy()
train_clean["review"] = clean_review_text(train_clean["review"])
train_clean = train_clean.drop_duplicates(subset="review", keep="first").reset_index(drop=True)

test_clean = test.copy()
test_clean["review"] = clean_review_text(test_clean["review"])

print(f"Training rows: {len(train):,} raw, {len(train_clean):,} after deduplication")
print(f"Test rows preserved: {len(test_clean):,}")

## Train the selected final model

The experiments selected bigram TF-IDF with at most 100,000 features and logistic regression with `C=2`.

In [ ]:
RANDOM_STATE = 42

final_model = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_features=100_000,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                C=2.0,
                max_iter=1_000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

final_model.fit(train_clean["review"], train_clean["sentiment"])
print("Final model fitted.")

## Generate and validate `submission.csv`

In [ ]:
submission = pd.DataFrame(
    {
        "id": test_clean["id"],
        "sentiment": final_model.predict_proba(test_clean["review"])[:, 1],
    }
)

assert list(submission.columns) == ["id", "sentiment"]
assert len(submission) == len(test)
assert submission["id"].is_unique
assert not submission.isna().any().any()
assert submission["sentiment"].between(0, 1).all()

output_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
submission_path = output_dir / "submission.csv"
submission.to_csv(submission_path, index=False)

print(f"Saved {len(submission):,} predictions to {submission_path}")
submission.head()